# Notebook 2 - Dataset Construction

## Objectives

Verify and create datasets necessary for the data processing step for the ML models. This would include creating rolling window dataFrames to create Path Signatures

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
DATA_DIR = Path("../data")

DATA_DIR.mkdir(exist_ok=True)

In [4]:
DATA = {
    "monthly_portfolios": DATA_DIR / "25_Portfolios_5x5_Usable_Monthly.csv",
    "daily_portfolios": DATA_DIR / "25_Portfolios_5x5_Usable_Daily.csv",

    "monthly_ff3": DATA_DIR / "F-F-3_Research_Usable_Monthly.csv",
    "daily_ff3": DATA_DIR / "F-F-3_Research_Usable_Daily.csv",

    "monthly_ff5": DATA_DIR / "F-F-5_Research_Usable_Monthly.csv",
    "daily_ff5": DATA_DIR / "F-F-5_Research_Usable_Daily.csv",
}

In [5]:
def load_dataset(path, frequency):
    df = pd.read_csv(path)

    if frequency == "monthly":
        df["Date"] = pd.to_datetime(df["Date"], format="%Y%m")

    else:
        df["Date"] = pd.to_datetime(df["Date"], format="%Y%m%d")

    return df

In [6]:
df = pd.read_csv(DATA["monthly_portfolios"])

In [7]:
monthly_returns = load_dataset(DATA["monthly_portfolios"], "monthly")
daily_returns = load_dataset(DATA["daily_portfolios"], "daily")

monthly_ff3 = load_dataset(DATA["monthly_ff3"], "monthly")
daily_ff3 = load_dataset(DATA["daily_ff3"], "daily")

monthly_ff5 = load_dataset(DATA["monthly_ff5"], "monthly")
daily_ff5 = load_dataset(DATA["daily_ff5"], "daily")

In [8]:
def merge_returns_and_factors(returns_df, factors_df):
    merged = returns_df.merge(factors_df, on="Date", how="inner")

    portfolio_cols = [col for col in returns_df.columns if col != "Date"]
    factor_cols = [col for col in factors_df.columns if col != "Date"]

    return merged, portfolio_cols, factor_cols

In [9]:
monthly_ff3_merged, monthly_portfolio_cols, monthly_ff3_factor_cols = merge_returns_and_factors(
    monthly_returns,
    monthly_ff3
)

monthly_ff5_merged, _, monthly_ff5_factor_cols = merge_returns_and_factors(
    monthly_returns,
    monthly_ff5
)

daily_ff3_merged, daily_portfolio_cols, daily_ff3_factor_cols = merge_returns_and_factors(
    daily_returns,
    daily_ff3
)

daily_ff5_merged, _, daily_ff5_factor_cols = merge_returns_and_factors(
    daily_returns,
    daily_ff5
)

In [10]:
merged_datasets = {
    "monthly_ff3": monthly_ff3_merged,
    "monthly_ff5": monthly_ff5_merged,
    "daily_ff3": daily_ff3_merged,
    "daily_ff5": daily_ff5_merged,
}

for name, df in merged_datasets.items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    print("Shape:", df.shape)
    print("Date Range:", df["Date"].min(), "->", df["Date"].max())
    print("Missing Values:", df.isna().sum().sum())
    display(df.head())

monthly_ff3
Shape: (1198, 30)
Date Range: 1926-07-01 00:00:00 -> 2026-04-01 00:00:00
Missing Values: 0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,Mkt-RF,SMB,HML,RF
0,1926-07-01,5.8276,-1.7006,0.5118,-2.1477,1.9583,1.2118,2.4107,0.6056,-2.6082,...,2.4678,3.3248,6.0909,2.0285,3.1263,0.5623,2.89,-2.55,-2.39,0.22
1,1926-08-01,-2.0206,-8.0282,1.3968,2.1483,8.5104,2.3620,-0.7525,3.8984,0.2299,...,5.3422,1.0169,4.1975,1.9769,5.4924,7.7576,2.64,-1.14,3.81,0.25
2,1926-09-01,-4.8291,-2.6806,-4.3417,-3.2683,0.8586,-2.6849,-0.5252,1.0789,-3.2877,...,0.8730,-1.2951,3.6610,0.1384,-0.7497,-2.4284,0.38,-1.36,0.05,0.23
3,1926-10-01,-9.3633,-3.5519,-3.5024,3.4413,-2.5452,-2.8014,-4.4191,-5.0767,-8.0271,...,-5.3525,-2.7382,-3.0061,-2.2467,-4.6725,-5.8129,-3.27,-0.14,0.82,0.32
4,1926-11-01,5.5888,4.1877,2.4384,-4.4495,0.5110,3.1023,-1.7317,3.0425,4.9538,...,1.8213,4.4331,2.5355,1.5280,3.6596,2.5636,2.54,-0.11,-0.61,0.31


monthly_ff5
Shape: (754, 32)
Date Range: 1963-07-01 00:00:00 -> 2026-04-01 00:00:00
Missing Values: 0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,Mkt-RF,SMB,HML,RMW,CMA,RF
0,1963-07-01,1.1287,-0.3632,0.7223,-0.0413,-1.2447,-1.8076,0.1929,-1.0149,-1.9749,...,0.4839,1.1360,-0.4285,-1.1045,-0.39,-0.48,-0.81,0.64,-1.15,0.27
1,1963-08-01,4.2396,1.3730,1.4917,2.5068,4.6644,5.5703,4.5220,4.4450,4.4662,...,4.2633,4.6341,8.1704,6.3984,5.08,-0.80,1.70,0.40,-0.38,0.25
2,1963-09-01,-1.7343,0.6204,-1.0007,-1.5215,-0.3584,-4.0525,-1.5072,-0.8638,-1.4935,...,-0.8081,-0.8497,-0.1912,-3.5033,-1.57,-0.43,0.00,-0.78,0.15,0.27
3,1963-10-01,0.3778,-0.7329,1.3066,0.1904,2.3711,1.1926,4.2411,2.3526,2.3058,...,1.7420,-0.3354,2.4176,0.4702,2.54,-1.34,-0.04,2.79,-2.25,0.29
4,1963-11-01,-3.3319,-3.8436,-1.7893,-1.0535,-1.1077,-4.2596,-1.7484,-0.7845,-0.0554,...,1.0080,-1.6914,-2.1316,1.3496,-0.86,-0.85,1.73,-0.43,2.27,0.27


daily_ff3
Shape: (26252, 30)
Date Range: 1926-07-01 00:00:00 -> 2026-05-28 00:00:00
Missing Values: 0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,Mkt-RF,SMB,HML,RF
0,1926-07-01,-0.46,0.72,0.85,0.30,-0.57,0.34,1.73,-0.02,-0.55,...,0.41,0.30,-0.06,0.44,-0.31,0.40,0.09,-0.25,-0.27,0.01
1,1926-07-02,0.57,0.77,-1.98,-0.41,-0.52,0.07,-0.06,-0.46,-0.03,...,0.56,0.50,0.60,0.33,0.51,0.24,0.45,-0.33,-0.06,0.01
2,1926-07-06,0.38,-0.46,-0.77,1.48,-0.28,-0.39,-0.04,0.35,0.07,...,-0.44,0.19,0.41,-0.12,-0.23,0.33,0.17,0.30,-0.39,0.01
3,1926-07-07,-0.81,-1.18,1.26,0.88,-0.61,0.63,-0.59,-0.84,-1.31,...,-0.08,-0.05,0.39,0.09,-0.16,1.53,0.09,-0.58,0.02,0.01
4,1926-07-08,0.56,-0.13,-1.10,-1.57,0.33,0.49,0.81,0.30,0.17,...,1.27,0.41,0.36,0.01,0.16,0.22,0.22,-0.38,0.19,0.01


daily_ff5
Shape: (15832, 32)
Date Range: 1963-07-01 00:00:00 -> 2026-05-28 00:00:00
Missing Values: 0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,Mkt-RF,SMB,HML,RMW,CMA,RF
0,1963-07-01,-0.51,-0.81,-0.65,-0.56,-0.71,-0.87,-0.70,-0.37,-0.47,...,-0.70,-0.60,-0.67,-1.30,-0.67,0.00,-0.34,-0.01,0.16,0.01
1,1963-07-02,0.51,0.92,0.28,0.59,0.62,0.72,0.24,0.77,0.62,...,0.82,0.81,0.86,1.38,0.79,-0.26,0.26,-0.07,-0.20,0.01
2,1963-07-03,0.80,0.52,0.71,0.48,0.35,0.46,0.60,0.54,0.36,...,0.60,0.97,0.66,0.38,0.63,-0.17,-0.09,0.18,-0.34,0.01
3,1963-07-05,0.29,0.21,0.70,0.44,0.41,0.77,0.73,0.41,0.18,...,0.20,1.04,0.15,0.18,0.40,0.08,-0.27,0.09,-0.34,0.01
4,1963-07-08,-0.42,-0.23,-0.64,-0.26,-0.49,-0.87,-0.92,-0.71,-0.56,...,-0.44,-0.37,-0.82,-1.09,-0.63,0.04,-0.18,-0.29,0.14,0.01


In [11]:
def create_top_labels(returns_df, top_pct=0.20):
    df = returns_df.copy()

    # Portfolio columns
    portfolio_cols = [col for col in df.columns if col != "Date"]

    # Next-period returns become today's target
    future_returns = df[portfolio_cols].shift(-1)

    # Rank portfolios within each date
    ranks = future_returns.rank(
        axis=1,
        ascending=False,
        method="min"
    )

    # Number of positive portfolios
    n_portfolios = len(portfolio_cols)
    top_n = max(1, int(np.ceil(n_portfolios * top_pct)))

    # Binary labels
    labels = (ranks <= top_n).astype(int)

    # Add dates
    labels.insert(0, "Date", df["Date"])

    return labels

In [12]:
monthly_labels = create_top_labels(monthly_returns)
daily_labels = create_top_labels(daily_returns)

display(monthly_labels.head())
display(daily_labels.head())

,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
0,1926-07-01,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1,1926-08-01,0,0,0,0,0,0,0,1,0,...,1,0,0,1,0,0,1,0,0,0
2,1926-09-01,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,1926-10-01,1,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
4,1926-11-01,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
0,1926-07-01,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
1,1926-07-02,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,1926-07-06,0,0,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,1926-07-07,1,0,0,0,0,1,1,0,0,...,0,0,0,1,1,0,0,0,0,0
4,1926-07-08,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [13]:
print("Monthly")
display(monthly_labels.head())

print("\nPositive labels per month:")
display(monthly_labels.drop(columns="Date").sum(axis=1).head())

print("\nDaily")
display(daily_labels.head())

print("\nPositive labels per day:")
display(daily_labels.drop(columns="Date").sum(axis=1).head())

Monthly


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
0,1926-07-01,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1,1926-08-01,0,0,0,0,0,0,0,1,0,...,1,0,0,1,0,0,1,0,0,0
2,1926-09-01,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,1926-10-01,1,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
4,1926-11-01,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0



Positive labels per month:


0    5
1    5
2    5
3    5
4    5
dtype: int64


Daily


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
0,1926-07-01,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
1,1926-07-02,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,1926-07-06,0,0,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,1926-07-07,1,0,0,0,0,1,1,0,0,...,0,0,0,1,1,0,0,0,0,0
4,1926-07-08,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0



Positive labels per day:


0    5
1    5
2    5
3    5
4    5
dtype: int64

In [14]:
def build_rolling_windows(
    merged_df,
    labels_df,
    window_size,
    portfolio_cols,
    factor_cols
):

    X = []
    y = []

    dates = []
    portfolios = []

    for i in range(window_size, len(merged_df) - 1):

        current_date = merged_df.loc[i, "Date"]

        for portfolio in portfolio_cols:

            # Window of portfolio return + factors
            window = merged_df.iloc[
                i-window_size:i
            ][[portfolio] + factor_cols].values

            label = labels_df.loc[i, portfolio]

            X.append(window)
            y.append(label)

            dates.append(current_date)
            portfolios.append(portfolio)

    meta = pd.DataFrame({
        "Date": dates,
        "Portfolio": portfolios
    })

    return (
        np.array(X),
        np.array(y),
        meta
    )

In [15]:
X_seq_monthly_ff5, y_monthly_ff5, meta_monthly_ff5 = build_rolling_windows(
    merged_df=monthly_ff5_merged,
    labels_df=monthly_labels,
    window_size=12,
    portfolio_cols=monthly_portfolio_cols,
    factor_cols=monthly_ff5_factor_cols
)

In [16]:
X_seq_daily_ff5, y_daily_ff5, meta_daily_ff5 = build_rolling_windows(
    merged_df=daily_ff5_merged,
    labels_df=daily_labels,
    window_size=60,
    portfolio_cols=daily_portfolio_cols,
    factor_cols=daily_ff5_factor_cols
)

In [17]:
print("Monthly FF5")
print(X_seq_monthly_ff5.shape)
print(y_monthly_ff5.shape)
display(meta_monthly_ff5.head())

print("\nDaily FF5")
print(X_seq_daily_ff5.shape)
print(y_daily_ff5.shape)
display(meta_daily_ff5.head())

Monthly FF5
(18525, 12, 7)
(18525,)


,Date,Portfolio
0,1964-07-01,SMALL LoBM
1,1964-07-01,ME1 BM2
2,1964-07-01,ME1 BM3
3,1964-07-01,ME1 BM4
4,1964-07-01,SMALL HiBM



Daily FF5
(394275, 60, 7)
(394275,)


,Date,Portfolio
0,1963-09-25,SMALL LoBM
1,1963-09-25,ME1 BM2
2,1963-09-25,ME1 BM3
3,1963-09-25,ME1 BM4
4,1963-09-25,SMALL HiBM


In [18]:
def chronological_split(X, y, meta, train_size=0.70, val_size=0.15):
    n = len(X)

    train_end = int(n * train_size)
    val_end = int(n * (train_size + val_size))

    X_train = X[:train_end]
    X_val = X[train_end:val_end]
    X_test = X[val_end:]

    y_train = y[:train_end]
    y_val = y[train_end:val_end]
    y_test = y[val_end:]

    meta_train = meta.iloc[:train_end]
    meta_val = meta.iloc[train_end:val_end]
    meta_test = meta.iloc[val_end:]

    return (
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        meta_train, meta_val, meta_test
    )

In [19]:
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    meta_train,
    meta_val,
    meta_test
) = chronological_split(
    X_seq_monthly_ff5,
    y_monthly_ff5,
    meta_monthly_ff5
)

In [20]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nTraining Dates")
print(meta_train["Date"].min(), "->", meta_train["Date"].max())

print("\nValidation Dates")
print(meta_val["Date"].min(), "->", meta_val["Date"].max())

print("\nTesting Dates")
print(meta_test["Date"].min(), "->", meta_test["Date"].max())

Training: (12967, 12, 7)
Validation: (2779, 12, 7)
Testing: (2779, 12, 7)

Training Dates
1964-07-01 00:00:00 -> 2007-09-01 00:00:00

Validation Dates
2007-09-01 00:00:00 -> 2016-12-01 00:00:00

Testing Dates
2016-12-01 00:00:00 -> 2026-03-01 00:00:00


In [21]:
np.save(DATA_DIR / "monthly_ff5_X.npy", X_train)
np.save(DATA_DIR / "monthly_ff5_y.npy", y_train)

meta_train.to_csv(
    DATA_DIR / "monthly_ff5_metadata.csv",
    index=False
)

In [22]:
def align_labels_to_merged(merged_df, labels_df):
    aligned = merged_df[["Date"]].merge(
        labels_df,
        on="Date",
        how="left"
    )

    return aligned

In [23]:
monthly_ff3_labels = align_labels_to_merged(
    monthly_ff3_merged,
    monthly_labels
)

monthly_ff5_labels = align_labels_to_merged(
    monthly_ff5_merged,
    monthly_labels
)

daily_ff3_labels = align_labels_to_merged(
    daily_ff3_merged,
    daily_labels
)

daily_ff5_labels = align_labels_to_merged(
    daily_ff5_merged,
    daily_labels
)

In [24]:
def build_rolling_windows(
    merged_df,
    labels_df,
    window_size,
    portfolio_cols,
    factor_cols
):
    X = []
    y = []
    metadata = []

    merged_df = merged_df.sort_values("Date").reset_index(drop=True)
    labels_df = labels_df.sort_values("Date").reset_index(drop=True)

    for i in range(window_size, len(merged_df) - 1):
        prediction_date = merged_df.loc[i, "Date"]
        realization_date = merged_df.loc[i + 1, "Date"]

        for portfolio in portfolio_cols:
            feature_cols = [portfolio] + factor_cols

            window = merged_df.loc[
                i - window_size:i - 1,
                feature_cols
            ].to_numpy(dtype=np.float32)

            label = labels_df.loc[i, portfolio]
            realized_next_return = merged_df.loc[i + 1, portfolio]

            if (
                np.isnan(window).any()
                or pd.isna(label)
                or pd.isna(realized_next_return)
            ):
                continue

            X.append(window)
            y.append(int(label))

            metadata.append({
                "Date": prediction_date,
                "RealizationDate": realization_date,
                "Portfolio": portfolio,
                "RealizedNextReturn": float(realized_next_return)
            })

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int8),
        pd.DataFrame(metadata)
    )

In [25]:
MONTHLY_WINDOW = 12
DAILY_WINDOW = 60

In [26]:
X_monthly_ff3, y_monthly_ff3, meta_monthly_ff3 = build_rolling_windows(
    merged_df=monthly_ff3_merged,
    labels_df=monthly_ff3_labels,
    window_size=MONTHLY_WINDOW,
    portfolio_cols=monthly_portfolio_cols,
    factor_cols=monthly_ff3_factor_cols
)

X_monthly_ff5, y_monthly_ff5, meta_monthly_ff5 = build_rolling_windows(
    merged_df=monthly_ff5_merged,
    labels_df=monthly_ff5_labels,
    window_size=MONTHLY_WINDOW,
    portfolio_cols=monthly_portfolio_cols,
    factor_cols=monthly_ff5_factor_cols
)

X_daily_ff3, y_daily_ff3, meta_daily_ff3 = build_rolling_windows(
    merged_df=daily_ff3_merged,
    labels_df=daily_ff3_labels,
    window_size=DAILY_WINDOW,
    portfolio_cols=daily_portfolio_cols,
    factor_cols=daily_ff3_factor_cols
)

X_daily_ff5, y_daily_ff5, meta_daily_ff5 = build_rolling_windows(
    merged_df=daily_ff5_merged,
    labels_df=daily_ff5_labels,
    window_size=DAILY_WINDOW,
    portfolio_cols=daily_portfolio_cols,
    factor_cols=daily_ff5_factor_cols
)

In [27]:
constructed_datasets = {
    "monthly_ff3": {
        "X": X_monthly_ff3,
        "y": y_monthly_ff3,
        "meta": meta_monthly_ff3
    },
    "monthly_ff5": {
        "X": X_monthly_ff5,
        "y": y_monthly_ff5,
        "meta": meta_monthly_ff5
    },
    "daily_ff3": {
        "X": X_daily_ff3,
        "y": y_daily_ff3,
        "meta": meta_daily_ff3
    },
    "daily_ff5": {
        "X": X_daily_ff5,
        "y": y_daily_ff5,
        "meta": meta_daily_ff5
    }
}

In [28]:
for name, dataset in constructed_datasets.items():
    X = dataset["X"]
    y = dataset["y"]
    meta = dataset["meta"]

    print("=" * 70)
    print(name)
    print("=" * 70)

    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Metadata shape:", meta.shape)

    print("Date range:", meta["Date"].min(), "->", meta["Date"].max())
    print("Number of portfolios:", meta["Portfolio"].nunique())

    print("Positive labels:", int(y.sum()))
    print("Positive-label rate:", round(float(y.mean()), 4))

    print("Missing X values:", int(np.isnan(X).sum()))
    print("Missing y values:", int(pd.isna(y).sum()))
    print()

monthly_ff3
X shape: (29625, 12, 5)
y shape: (29625,)
Metadata shape: (29625, 4)
Date range: 1927-07-01 00:00:00 -> 2026-03-01 00:00:00
Number of portfolios: 25
Positive labels: 5925
Positive-label rate: 0.2
Missing X values: 0
Missing y values: 0

monthly_ff5
X shape: (18525, 12, 7)
y shape: (18525,)
Metadata shape: (18525, 4)
Date range: 1964-07-01 00:00:00 -> 2026-03-01 00:00:00
Number of portfolios: 25
Positive labels: 3705
Positive-label rate: 0.2
Missing X values: 0
Missing y values: 0

daily_ff3
X shape: (654775, 60, 5)
y shape: (654775,)
Metadata shape: (654775, 4)
Date range: 1926-09-14 00:00:00 -> 2026-05-27 00:00:00
Number of portfolios: 25
Positive labels: 133148
Positive-label rate: 0.2033
Missing X values: 0
Missing y values: 0

daily_ff5
X shape: (394275, 60, 7)
y shape: (394275,)
Metadata shape: (394275, 4)
Date range: 1963-09-25 00:00:00 -> 2026-05-27 00:00:00
Number of portfolios: 25
Positive labels: 80356
Positive-label rate: 0.2038
Missing X values: 0
Missing y valu

In [29]:
dataset_name = "monthly_ff5"

X_example = constructed_datasets[dataset_name]["X"]
y_example = constructed_datasets[dataset_name]["y"]
meta_example = constructed_datasets[dataset_name]["meta"]

sample_number = 0

print("Metadata:")
display(meta_example.iloc[[sample_number]])

print("Target label:", y_example[sample_number])
print("Window shape:", X_example[sample_number].shape)

display(pd.DataFrame(X_example[sample_number]))

Metadata:


,Date,RealizationDate,Portfolio,RealizedNextReturn
0,1964-07-01,1964-08-01,SMALL LoBM,1.6973


Target label: 1
Window shape: (12, 7)


,0,1,2,3,4,5,6
0,1.1287,-0.39,-0.48,-0.81,0.64,-1.15,0.27
1,4.2396,5.08,-0.80,1.70,0.40,-0.38,0.25
2,-1.7343,-1.57,-0.43,0.00,-0.78,0.15,0.27
3,0.3778,2.54,-1.34,-0.04,2.79,-2.25,0.29
4,-3.3319,-0.86,-0.85,1.73,-0.43,2.27,0.27
5,-2.3435,1.83,-1.89,-0.21,0.12,-0.25,0.29
6,3.8925,2.27,0.10,1.63,0.21,1.48,0.30
7,2.8826,1.55,0.33,2.81,0.11,0.81,0.26
8,0.4753,1.41,1.41,3.29,-2.03,2.98,0.31
9,-1.5767,0.11,-1.48,-0.54,-1.32,-1.13,0.29


In [30]:
def chronological_date_split(
    X,
    y,
    meta,
    train_fraction=0.70,
    validation_fraction=0.15
):
    """
    Split observations using unique dates.

    Every portfolio belonging to a given date remains in the same split.
    """

    unique_dates = np.array(sorted(meta["Date"].unique()))

    n_dates = len(unique_dates)

    train_date_end = int(n_dates * train_fraction)
    validation_date_end = int(
        n_dates * (train_fraction + validation_fraction)
    )

    train_dates = unique_dates[:train_date_end]
    validation_dates = unique_dates[
        train_date_end:validation_date_end
    ]
    test_dates = unique_dates[validation_date_end:]

    train_mask = meta["Date"].isin(train_dates).to_numpy()
    validation_mask = meta["Date"].isin(validation_dates).to_numpy()
    test_mask = meta["Date"].isin(test_dates).to_numpy()

    splits = {
        "train": {
            "X": X[train_mask],
            "y": y[train_mask],
            "meta": meta.loc[train_mask].reset_index(drop=True)
        },
        "validation": {
            "X": X[validation_mask],
            "y": y[validation_mask],
            "meta": meta.loc[validation_mask].reset_index(drop=True)
        },
        "test": {
            "X": X[test_mask],
            "y": y[test_mask],
            "meta": meta.loc[test_mask].reset_index(drop=True)
        }
    }

    return splits

In [31]:
all_splits = {}

for name, dataset in constructed_datasets.items():
    all_splits[name] = chronological_date_split(
        X=dataset["X"],
        y=dataset["y"],
        meta=dataset["meta"],
        train_fraction=0.70,
        validation_fraction=0.15
    )

In [32]:
for dataset_name, splits in all_splits.items():
    print("=" * 70)
    print(dataset_name)
    print("=" * 70)

    for split_name, split_data in splits.items():
        X_split = split_data["X"]
        y_split = split_data["y"]
        meta_split = split_data["meta"]

        print(
            f"{split_name:10} | "
            f"X: {X_split.shape} | "
            f"y: {y_split.shape} | "
            f"dates: {meta_split['Date'].min()} "
            f"-> {meta_split['Date'].max()} | "
            f"positive rate: {y_split.mean():.4f}"
        )

    print()

monthly_ff3
train      | X: (20725, 12, 5) | y: (20725,) | dates: 1927-07-01 00:00:00 -> 1996-07-01 00:00:00 | positive rate: 0.2000
validation | X: (4450, 12, 5) | y: (4450,) | dates: 1996-08-01 00:00:00 -> 2011-05-01 00:00:00 | positive rate: 0.2000
test       | X: (4450, 12, 5) | y: (4450,) | dates: 2011-06-01 00:00:00 -> 2026-03-01 00:00:00 | positive rate: 0.2000

monthly_ff5
train      | X: (12950, 12, 7) | y: (12950,) | dates: 1964-07-01 00:00:00 -> 2007-08-01 00:00:00 | positive rate: 0.2000
validation | X: (2775, 12, 7) | y: (2775,) | dates: 2007-09-01 00:00:00 -> 2016-11-01 00:00:00 | positive rate: 0.2000
test       | X: (2800, 12, 7) | y: (2800,) | dates: 2016-12-01 00:00:00 -> 2026-03-01 00:00:00 | positive rate: 0.2000

daily_ff3
train      | X: (458325, 60, 5) | y: (458325,) | dates: 1926-09-14 00:00:00 -> 1995-03-06 00:00:00 | positive rate: 0.2033
validation | X: (98225, 60, 5) | y: (98225,) | dates: 1995-03-07 00:00:00 -> 2010-10-11 00:00:00 | positive rate: 0.2035
te

In [33]:
for dataset_name, splits in all_splits.items():

    train_dates = set(splits["train"]["meta"]["Date"])
    validation_dates = set(splits["validation"]["meta"]["Date"])
    test_dates = set(splits["test"]["meta"]["Date"])

    assert train_dates.isdisjoint(validation_dates)
    assert train_dates.isdisjoint(test_dates)
    assert validation_dates.isdisjoint(test_dates)

    print(dataset_name, "passed the date-overlap check.")

monthly_ff3 passed the date-overlap check.
monthly_ff5 passed the date-overlap check.
daily_ff3 passed the date-overlap check.
daily_ff5 passed the date-overlap check.


In [34]:
DATASET_DIR = Path("../data/datasets")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", DATASET_DIR.resolve())
print("Folder exists:", DATASET_DIR.exists())

Output folder: C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\data\datasets
Folder exists: True


In [35]:
for dataset_name, splits in all_splits.items():

    dataset_output_dir = DATASET_DIR / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)

    for split_name, split_data in splits.items():

        np.save(
            dataset_output_dir / f"X_{split_name}.npy",
            split_data["X"]
        )

        np.save(
            dataset_output_dir / f"y_{split_name}.npy",
            split_data["y"]
        )

        split_data["meta"].to_csv(
            dataset_output_dir / f"meta_{split_name}.csv",
            index=False
        )

    print(f"Saved {dataset_name} to {dataset_output_dir}")

Saved monthly_ff3 to ..\data\datasets\monthly_ff3
Saved monthly_ff5 to ..\data\datasets\monthly_ff5
Saved daily_ff3 to ..\data\datasets\daily_ff3
Saved daily_ff5 to ..\data\datasets\daily_ff5


In [36]:
for path in sorted(DATASET_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DATASET_DIR))

daily_ff3\meta_test.csv
daily_ff3\meta_train.csv
daily_ff3\meta_validation.csv
daily_ff3\X_test.npy
daily_ff3\X_train.npy
daily_ff3\X_validation.npy
daily_ff3\y_test.npy
daily_ff3\y_train.npy
daily_ff3\y_validation.npy
daily_ff5\meta_test.csv
daily_ff5\meta_train.csv
daily_ff5\meta_validation.csv
daily_ff5\X_test.npy
daily_ff5\X_train.npy
daily_ff5\X_validation.npy
daily_ff5\y_test.npy
daily_ff5\y_train.npy
daily_ff5\y_validation.npy
monthly_ff3\meta_test.csv
monthly_ff3\meta_train.csv
monthly_ff3\meta_validation.csv
monthly_ff3\X_test.npy
monthly_ff3\X_train.npy
monthly_ff3\X_validation.npy
monthly_ff3\y_test.npy
monthly_ff3\y_train.npy
monthly_ff3\y_validation.npy
monthly_ff5\meta_test.csv
monthly_ff5\meta_train.csv
monthly_ff5\meta_validation.csv
monthly_ff5\X_test.npy
monthly_ff5\X_train.npy
monthly_ff5\X_validation.npy
monthly_ff5\y_test.npy
monthly_ff5\y_train.npy
monthly_ff5\y_validation.npy


In [37]:
test_dir = DATASET_DIR / "monthly_ff5"

X_train_check = np.load(test_dir / "X_train.npy")
y_train_check = np.load(test_dir / "y_train.npy")
meta_train_check = pd.read_csv(
    test_dir / "meta_train.csv",
    parse_dates=["Date"]
)

print("X train:", X_train_check.shape)
print("y train:", y_train_check.shape)
print("Metadata:", meta_train_check.shape)

display(meta_train_check.head())

X train: (12950, 12, 7)
y train: (12950,)
Metadata: (12950, 4)


,Date,RealizationDate,Portfolio,RealizedNextReturn
0,1964-07-01,1964-08-01,SMALL LoBM,1.6973
1,1964-07-01,1964-08-01,ME1 BM2,-2.5518
2,1964-07-01,1964-08-01,ME1 BM3,-1.7484
3,1964-07-01,1964-08-01,ME1 BM4,-0.8049
4,1964-07-01,1964-08-01,SMALL HiBM,-0.4831
